In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 34. Week 24 — Distribution shift、nested evaluation、conformal境界

> intervalを計算できることと、そのcoverage theoremの仮定が成り立つことは別である。

## 学習目標

- inner selectionとouter evaluationを分離できる
- feature driftとperformance driftを別々に測れる
- split-conformal intervalをfinite-sample rankで構成できる
- exchangeabilityを依存時系列へ無条件に仮定しない

## 前提知識

- B5のtemporal validation
- quantileとcoverage
- Week 21–23のmodel familyとregime diagnostic

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 34


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)

assert treasury.quality.accepted
assert np.all(forecast.target_dates > forecast.prediction_dates)
assert np.all(np.isfinite(forecast.features))
crosses_methodology_break = (
    (forecast.prediction_dates < qt.TREASURY_METHOD_BREAK.to_datetime64())
    & (forecast.target_dates >= qt.TREASURY_METHOD_BREAK.to_datetime64())
)
assert not np.any(crosses_methodology_break)

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("rows / forecast rows:", len(rates), len(forecast.regression_target))
print("methodology-crossing targets retained:", int(crosses_methodology_break.sum()))
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
rows / forecast rows: 2750 2728
methodology-crossing targets retained: 0
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Nested temporal protocol

outer testを最後に固定する。alphaはproper-training候補期間のinner train/selectionだけで選び、
その後にproper trainingへfitする。外側のvalidation partitionはconformal calibration専用に残す。

In [4]:
split = qt.chronological_split(len(forecast.regression_target), gap=1)
features = forecast.features
target = forecast.regression_target

inner_boundary = int(np.floor(0.8 * split.train.size))
inner_training = split.train[: inner_boundary - forecast.horizon_publications]
inner_selection = split.train[inner_boundary:]
assert (
    forecast.target_dates[inner_training[-1]]
    < forecast.prediction_dates[inner_selection[0]]
)

alpha_grid = [0.1, 1.0, 10.0, 100.0]
inner_rows = []
for alpha in alpha_grid:
    model = qt.fit_ridge(features[inner_training], target[inner_training], alpha=alpha)
    metric = qt.regression_metrics(
        target[inner_selection],
        model.predict(features[inner_selection]),
    )
    inner_rows.append({"alpha": alpha, "inner_validation_rmse_bp": metric.rmse})
inner_table = pd.DataFrame(inner_rows)
selected_alpha = float(inner_table.loc[inner_table["inner_validation_rmse_bp"].idxmin(), "alpha"])
display(inner_table)
print("selected alpha before outer test:", selected_alpha)
print("inner train / selection rows:", len(inner_training), len(inner_selection))

,alpha,inner_validation_rmse_bp
0,0.1,3.890246
1,1.0,3.862121
2,10.0,3.845682
3,100.0,3.844958


selected alpha before outer test: 100.0
inner train / selection rows: 1307 327


## 2. Feature drift

standardized mean differenceはlocation shift、population stability index（PSI）はreference quantile binのshare shiftを見る。閾値は普遍的な合否ではなく、追加診断のsignalである。

In [5]:
drift = qt.feature_drift_report(
    features[split.train],
    features[split.test],
    feature_names=forecast.feature_names,
)
drift_table = pd.DataFrame(
    {
        "feature": drift.feature_names,
        "standardized_mean_difference": drift.standardized_mean_difference,
        "psi": drift.population_stability_index,
    }
).sort_values("psi", ascending=False)
display(drift_table.head(10))

fig = go.Figure(
    go.Scatter(
        x=drift_table["standardized_mean_difference"],
        y=drift_table["psi"],
        mode="markers+text",
        text=drift_table["feature"],
        textposition="top center",
    )
)
fig.update_layout(
    title="Train-to-test feature drift diagnostics",
    xaxis_title="Standardized mean difference",
    yaxis_title="Population stability index",
    template="plotly_white",
)
fig.show()

,feature,standardized_mean_difference,psi
0,yield_3m_pct,5.219630,16.563441
3,yield_10y_pct,4.559138,16.553117
5,curve_level_pct,4.808435,16.552599
1,yield_2y_pct,4.377000,16.519652
4,yield_30y_pct,4.682383,16.518342
2,yield_5y_pct,4.289600,16.501155
6,curve_slope_pct,-1.574392,8.998768
7,curve_curvature_pct,-1.911896,6.882993
14,10y_rolling_vol_20d_bp,0.897317,2.632008
12,curve_slope_change_lag1_bp,0.047647,0.251326


## 3. Split conformal interval

calibration residual $R_i=|Y_i-\hat f(X_i)|$ の有限標本higher quantileを使う。
$\hat f$ はproper trainingだけで一度fitし、calibration outcomeをrefitへ戻さない。
calibration rowsとtest rowsがexchangeableならmarginal coverage保証が得られるが、金融時系列では依存とshiftがある。ここではintervalを計算し、保証ではなくperiod別empirical coverageとerror driftを報告する。

calibration sizeを $n$、miscoverageを $\alpha$ とすると、sorted residualの

$$
k=\min\left\{\left\lceil(n+1)(1-\alpha)\right\rceil,n\right\}
$$

番目をhalf-widthにする。このrank correctionもexchangeabilityを置いたときのものである。

In [6]:
conformal_model = qt.fit_ridge(
    features[split.train],
    target[split.train],
    alpha=selected_alpha,
)
validation_prediction = conformal_model.predict(features[split.validation])
test_prediction = conformal_model.predict(features[split.test])
interval = qt.split_conformal_interval(
    target[split.validation],
    validation_prediction,
    test_prediction,
    miscoverage=0.1,
)
covered = (target[split.test] >= interval.lower) & (target[split.test] <= interval.upper)
interval_summary = pd.DataFrame(
    {
        "nominal_coverage": [interval.nominal_coverage],
        "empirical_coverage": [covered.mean()],
        "mean_width_bp": [np.mean(interval.upper - interval.lower)],
        "residual_quantile_bp": [interval.residual_quantile],
        "exchangeability_verified": [False],
    }
)
display(interval_summary)

period_rows = []
for period_name, period_indices in [
    ("test_first_half", split.test[: split.test.size // 2]),
    ("test_second_half", split.test[split.test.size // 2 :]),
]:
    local_prediction = conformal_model.predict(features[period_indices])
    local_position = np.searchsorted(split.test, period_indices)
    local_covered = (
        (target[period_indices] >= interval.lower[local_position])
        & (target[period_indices] <= interval.upper[local_position])
    )
    ridge_metric = qt.regression_metrics(target[period_indices], local_prediction)
    zero_metric = qt.regression_metrics(
        target[period_indices],
        np.zeros(period_indices.size),
    )
    period_rows.append(
        {
            "period": period_name,
            "rows": period_indices.size,
            "ridge_rmse_bp": ridge_metric.rmse,
            "zero_rmse_bp": zero_metric.rmse,
            "empirical_coverage": local_covered.mean(),
        }
    )
performance_drift_table = pd.DataFrame(period_rows)
display(performance_drift_table)

,nominal_coverage,empirical_coverage,mean_width_bp,residual_quantile_bp,exchangeability_verified
0,0.9,0.959707,23.90236,11.95118,False


,period,rows,ridge_rmse_bp,zero_rmse_bp,empirical_coverage
0,test_first_half,273,6.228040,6.230317,0.930403
1,test_second_half,273,4.888117,4.888126,0.989011


In [7]:
display_indices = np.arange(min(120, split.test.size))
dates = pd.to_datetime(forecast.prediction_dates[split.test][display_indices])
fig = go.Figure()
fig.add_scatter(
    x=np.r_[dates, dates[::-1]],
    y=np.r_[interval.upper[display_indices], interval.lower[display_indices][::-1]],
    fill="toself",
    line={"color": "rgba(0,0,0,0)"},
    fillcolor="rgba(245,133,24,0.2)",
    name="split-conformal interval",
)
fig.add_scatter(x=dates, y=test_prediction[display_indices], name="ridge", mode="lines")
fig.add_scatter(
    x=dates,
    y=target[split.test][display_indices],
    name="actual",
    mode="lines",
    line={"color": "black", "width": 1},
)
fig.update_layout(
    title="Empirical interval audit under temporal dependence",
    xaxis_title="Prediction date",
    yaxis_title="Next-day change (bp)",
    template="plotly_white",
)
fig.show()

## 4. 失敗モード

- outer testでalphaを選ぶ
- feature driftだけを見てperformance driftを推測する
- PSIの慣用thresholdを普遍的な統計検定と呼ぶ
- dependent time seriesにexchangeabilityを無条件に置く
- aggregate coverageだけを報告し、period別undercoverageを隠す

## 5. 段階別演習

### 基礎

1. split-conformalのfinite-sample rankを導出せよ。
2. train/testのtop drift featuresを確認せよ。

### 標準

3. test前半・後半でcoverageを分けよ。
4. rolling calibration windowでinterval幅を更新せよ。

### 研究

5. block dependenceを考慮したcoverage診断を設計せよ。
6. adaptive conformalを実装する前に保証とestimandを定義せよ。

## 6. Exit Criteria

- [ ] inner selectionとouter evaluationを分離できる
- [ ] feature driftとerror driftを別々に報告できる
- [ ] conformal quantile rankを実装できる
- [ ] exchangeability未検証を明記できる
- [ ] empirical coverageをperiod別に監査できる

## 7. 出典

- [Lei et al., Distribution-Free Predictive Inference](https://doi.org/10.1080/01621459.2017.1307116) — split conformalの理論
- [Gibbs & Candès, Adaptive Conformal Inference Under Distribution Shift](https://papers.neurips.cc/paper_files/paper/2021/hash/0d441de75945e5acbc865406fc9a2559-Abstract.html) — shift下の拡張
- [scikit-learn Nested CV Example](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html) — selectionとevaluationの分離